In [1]:
import psycopg2
import json
import os
import glob
from dotenv import load_dotenv

load_dotenv()

conn = psycopg2.connect(
    host="localhost",
    port=5432,
    database="ESG_DB",
    user="postgres",
    password=os.environ["DB_PASSWORD"]
)
cur = conn.cursor()
print("Connected to PostgreSQL!")

Connected to PostgreSQL!


In [2]:
cur.execute("""
    CREATE TABLE IF NOT EXISTS companies (
        id SERIAL PRIMARY KEY,
        name VARCHAR(100) UNIQUE NOT NULL,
        region VARCHAR(10),
        country VARCHAR(50),
        sector VARCHAR(50)
    );
""")

cur.execute("""
    CREATE TABLE IF NOT EXISTS filings (
        id SERIAL PRIMARY KEY,
        company_id INTEGER REFERENCES companies(id),
        filing_type VARCHAR(10),
        filing_date DATE,
        file_path VARCHAR(200)
    );
""")

cur.execute("""
    CREATE TABLE IF NOT EXISTS chunks (
        id SERIAL PRIMARY KEY,
        filing_id INTEGER REFERENCES filings(id),
        chunk_index INTEGER,
        text TEXT,
        word_count INTEGER
    );
""")

cur.execute("""
    CREATE TABLE IF NOT EXISTS esg_scores (
        id SERIAL PRIMARY KEY,
        filing_id INTEGER REFERENCES filings(id),
        pillar VARCHAR(1),
        subcategory VARCHAR(50),
        score INTEGER,
        justification TEXT,
        evidence_quote TEXT,
        model_used VARCHAR(50),
        created_at TIMESTAMP DEFAULT NOW()
    );
""")

conn.commit()
print("All 4 tables created!")

All 4 tables created!


In [3]:
companies = [
    # Banks
    ("JPMorgan",           "US", "United States",  "Bank"),
    ("Goldman",            "US", "United States",  "Bank"),
    ("BankOfAmerica",      "US", "United States",  "Bank"),
    ("WellsFargo",         "US", "United States",  "Bank"),
    ("Citigroup",          "US", "United States",  "Bank"),
    ("MorganStanley",      "US", "United States",  "Bank"),
    ("USBancorp",          "US", "United States",  "Bank"),
    ("PNCFinancial",       "US", "United States",  "Bank"),
    # Capital management
    ("BlackRock",          "US", "United States",  "Capital Management"),
    ("StateStreet",        "US", "United States",  "Capital Management"),
    ("CharlesSchwab",      "US", "United States",  "Capital Management"),
    ("BerkshireHathaway",  "US", "United States",  "Capital Management"),
    ("TRowePrice",         "US", "United States",  "Capital Management"),
    # Insurance
    ("MetLife",            "US", "United States",  "Insurance"),
    ("PrudentialFinancial","US", "United States",  "Insurance"),
    ("Aflac",              "US", "United States",  "Insurance"),
    ("Allstate",           "US", "United States",  "Insurance"),
    ("Travelers",          "US", "United States",  "Insurance"),
    # Fintech
    ("Visa",               "US", "United States",  "Fintech"),
    ("Mastercard",         "US", "United States",  "Fintech"),
    ("PayPal",             "US", "United States",  "Fintech"),
    ("Fiserv",             "US", "United States",  "Fintech"),
    # Credit
    ("AmericanExpress",    "US", "United States",  "Credit"),
    ("CapitalOne",         "US", "United States",  "Credit"),
    ("DiscoverFinancial",  "US", "United States",  "Credit"),
    # Exchange/Infrastructure
    ("ICE",                "US", "United States",  "Exchange"),
    ("Nasdaq",             "US", "United States",  "Exchange"),
    ("CMEGroup",           "US", "United States",  "Exchange"),
]

for company in companies:
    cur.execute("""
        INSERT INTO companies (name, region, country, sector)
        VALUES (%s, %s, %s, %s)
        ON CONFLICT (name) DO NOTHING;
    """, company)

conn.commit()
print(f"Inserted {len(companies)} companies!")

Inserted 28 companies!
